In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression

from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    roc_auc_score
)

print("Libraries imported successfully")

In [ ]:
df = pd.read_csv("cleaned_data.csv")

print("Shape:", df.shape)

display(df.head())

df.info()

In [ ]:
# Regression target
y_reg = df["SalePrice"]


# Classification target
median_price = y_reg.median()

y_clf = (y_reg > median_price).astype(int)


# Features
X = df.drop("SalePrice", axis=1)


print("X shape:", X.shape)

print("\nRegression Target:")
print(y_reg.head())

print("\nClassification Target:")
print(y_clf.value_counts())

In [ ]:
cat_cols = X.select_dtypes(include=["object"]).columns

num_cols = X.select_dtypes(include=["number"]).columns


print("Categorical columns:")
print(list(cat_cols))

print("\nNumerical columns:")
print(list(num_cols))

In [ ]:
X_encoded = pd.get_dummies(
    X,
    columns=cat_cols,
    drop_first=True
)

print("Encoded Shape:", X_encoded.shape)

In [ ]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_encoded,
    y_reg,
    test_size=0.2,
    random_state=42
)


X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_encoded,
    y_clf,
    test_size=0.2,
    random_state=42
)


print(X_train_reg.shape)
print(X_test_reg.shape)

In [ ]:
scaler = StandardScaler()

X_train_reg_scaled = scaler.fit_transform(X_train_reg)

X_test_reg_scaled = scaler.transform(X_test_reg)


X_train_clf_scaled = scaler.fit_transform(X_train_clf)

X_test_clf_scaled = scaler.transform(X_test_clf)


print("Scaling completed")

In [ ]:
# ======================================================
# Linear Regression Model
# ======================================================

lr_model = LinearRegression()

lr_model.fit(
    X_train_reg_scaled,
    y_train_reg
)

y_pred_reg = lr_model.predict(
    X_test_reg_scaled
)

mse_lr = mean_squared_error(
    y_test_reg,
    y_pred_reg
)

r2_lr = r2_score(
    y_test_reg,
    y_pred_reg
)


print("Linear Regression Results")
print("-------------------------")
print("MSE:", mse_lr)
print("R2 Score:", r2_lr)

In [ ]:
# Coefficient table

coef_df = pd.DataFrame({
    "Feature": X_encoded.columns,
    "Coefficient": lr_model.coef_
})


coef_df["Absolute_Coefficient"] = (
    coef_df["Coefficient"].abs()
)


top_coefficients = (
    coef_df
    .sort_values(
        by="Absolute_Coefficient",
        ascending=False
    )
    .head(10)
)


display(top_coefficients)

In [ ]:
from sklearn.linear_model import Ridge


ridge_model = Ridge(alpha=1.0)


ridge_model.fit(
    X_train_reg_scaled,
    y_train_reg
)


y_pred_ridge = ridge_model.predict(
    X_test_reg_scaled
)


mse_ridge = mean_squared_error(
    y_test_reg,
    y_pred_ridge
)

r2_ridge = r2_score(
    y_test_reg,
    y_pred_ridge
)


print("Ridge Regression Results")
print("------------------------")
print("MSE:", mse_ridge)
print("R2 Score:", r2_ridge)

In [ ]:
regression_results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Ridge Regression"
    ],
    "MSE": [
        mse_lr,
        mse_ridge
    ],
    "R2": [
        r2_lr,
        r2_ridge
    ]
})


display(regression_results)

In [ ]:
coef_df = pd.DataFrame({
    "Feature": X_encoded.columns,
    "Coefficient": lr_model.coef_
})

coef_df["Absolute_Coefficient"] = coef_df["Coefficient"].abs()

top_coefficients = (
    coef_df
    .sort_values(
        by="Absolute_Coefficient",
        ascending=False
    )
    .head(10)
)

display(top_coefficients)

In [ ]:
print("Training class distribution:")
print(y_train_clf.value_counts())

print("\nTest class distribution:")
print(y_test_clf.value_counts())

In [ ]:
log_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

log_model.fit(
    X_train_clf_scaled,
    y_train_clf
)

y_pred_clf = log_model.predict(
    X_test_clf_scaled
)

y_prob_clf = log_model.predict_proba(
    X_test_clf_scaled
)[:,1]

print("Logistic Regression trained")

In [ ]:
# ======================================================
# Logistic Regression Model
# ======================================================

log_model = LogisticRegression(
    max_iter=1000
)


log_model.fit(
    X_train_clf_scaled,
    y_train_clf
)


# Predictions

y_pred_clf = log_model.predict(
    X_test_clf_scaled
)


# Probabilities for ROC

y_prob_clf = log_model.predict_proba(
    X_test_clf_scaled
)[:,1]


print("Logistic Regression trained successfully")

In [ ]:
# Confusion Matrix

cm = confusion_matrix(
    y_test_clf,
    y_pred_clf
)

print("Confusion Matrix:")
print(cm)


print("\nClassification Report:")
print(
    classification_report(
        y_test_clf,
        y_pred_clf
    )
)


accuracy = accuracy_score(
    y_test_clf,
    y_pred_clf
)

precision = precision_score(
    y_test_clf,
    y_pred_clf
)

recall = recall_score(
    y_test_clf,
    y_pred_clf
)

f1 = f1_score(
    y_test_clf,
    y_pred_clf
)


print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

In [ ]:
# ROC Curve

fpr, tpr, thresholds = roc_curve(
    y_test_clf,
    y_prob_clf
)


auc_score = roc_auc_score(
    y_test_clf,
    y_prob_clf
)


plt.figure(figsize=(8,6))

plt.plot(
    fpr,
    tpr,
    label=f"AUC = {auc_score:.3f}"
)


plt.plot(
    [0,1],
    [0,1],
    linestyle="--"
)


plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title(
    "ROC Curve - Logistic Regression"
)

plt.legend()

plt.grid()

plt.savefig(
    "plots/roc_curve.png",
    bbox_inches="tight"
)

plt.show()


print("AUC:", auc_score)

In [ ]:
# ======================================================
# Decision Threshold Sensitivity
# ======================================================

threshold_results = []


thresholds = np.arange(
    0.30,
    0.71,
    0.10
)


for threshold in thresholds:

    y_threshold_pred = (
        y_prob_clf >= threshold
    ).astype(int)


    precision = precision_score(
        y_test_clf,
        y_threshold_pred
    )


    recall = recall_score(
        y_test_clf,
        y_threshold_pred
    )


    f1 = f1_score(
        y_test_clf,
        y_threshold_pred
    )


    threshold_results.append(
        [
            threshold,
            precision,
            recall,
            f1
        ]
    )


threshold_df = pd.DataFrame(
    threshold_results,
    columns=[
        "Threshold",
        "Precision",
        "Recall",
        "F1"
    ]
)


display(threshold_df)

In [ ]:
# ======================================================
# Logistic Regression with C=0.01
# ======================================================

log_model_c001 = LogisticRegression(
    C=0.01,
    max_iter=1000
)


log_model_c001.fit(
    X_train_clf_scaled,
    y_train_clf
)


y_pred_c001 = log_model_c001.predict(
    X_test_clf_scaled
)


y_prob_c001 = log_model_c001.predict_proba(
    X_test_clf_scaled
)[:,1]


precision_c001 = precision_score(
    y_test_clf,
    y_pred_c001
)

recall_c001 = recall_score(
    y_test_clf,
    y_pred_c001
)


auc_c001 = roc_auc_score(
    y_test_clf,
    y_prob_c001
)


print("C=0.01 Logistic Regression")
print("--------------------------")
print("Precision:", precision_c001)
print("Recall:", recall_c001)
print("AUC:", auc_c001)

In [ ]:
regularization_results = pd.DataFrame({

    "Model":[
        "Logistic Regression C=1.0",
        "Logistic Regression C=0.01"
    ],

    "Precision":[
        precision,
        precision_c001
    ],

    "Recall":[
        recall,
        recall_c001
    ],

    "AUC":[
        auc_score,
        auc_c001
    ]

})


display(regularization_results)

In [ ]:
# ======================================================
# Bootstrap Confidence Interval for AUC Difference
# ======================================================

np.random.seed(42)

bootstrap_differences = []


n_iterations = 500


for i in range(n_iterations):

    indices = np.random.choice(
        len(y_test_clf),
        size=len(y_test_clf),
        replace=True
    )


    y_sample = y_test_clf.iloc[indices]

    prob_c1_sample = y_prob_clf[indices]

    prob_c001_sample = y_prob_c001[indices]


    auc_c1 = roc_auc_score(
        y_sample,
        prob_c1_sample
    )


    auc_c001 = roc_auc_score(
        y_sample,
        prob_c001_sample
    )


    difference = (
        auc_c1 - auc_c001
    )


    bootstrap_differences.append(
        difference
    )


bootstrap_differences = np.array(
    bootstrap_differences
)


mean_difference = np.mean(
    bootstrap_differences
)


lower_ci = np.percentile(
    bootstrap_differences,
    2.5
)


upper_ci = np.percentile(
    bootstrap_differences,
    97.5
)


print("Mean AUC Difference:",
      mean_difference)

print("95% Confidence Interval:")
print(
    lower_ci,
    "to",
    upper_ci
)

In [ ]:
print("Final X shape:", X_encoded.shape)

print("Regression target:")
print(y_reg.describe())

print("\nClassification target:")
print(y_clf.value_counts())

print("\nNotebook completed successfully")